# Energy Efficiency Linear Regression

This notebook presents the complete Y1/Y2 workflow, including EDA, leakage-safe preprocessing, scratch gradient descent, cross-validation, diagnostics, and limitations.


## 1. Load and audit the dataset


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from data_loader import load_dataset, print_schema
from preprocessing import preprocess_pipeline, make_kfold_splits
from linear_regression import LinearRegressionScratch
from metrics import mae, rmse, r2_score

data_path = PROJECT_ROOT / 'data' / 'energy_efficiency_building_heating_cooling_load_dataset.csv'
df = load_dataset(data_path)
print_schema(df)
df.describe(include='all')


## 2. EDA for Heating Load and Cooling Load


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['Y1'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Heating Load (Y1)')
axes[1].hist(df['Y2'], bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Cooling Load (Y2)')
plt.tight_layout()
plt.show()

df[[f'X{i}' for i in range(1, 9)] + ['Y1', 'Y2']].corr(numeric_only=True)[['Y1', 'Y2']]


## 3. Multicollinearity and leakage controls

X2 is excluded because the dataset satisfies X2 = X3 + 2X4. The opposite target is removed separately for each target run.


In [ ]:
print('X2 - X3 - 2*X4 max residual:', np.abs(df['X2'] - df['X3'] - 2 * df['X4']).max())
print('Y1 run excludes Y2; Y2 run excludes Y1.')


## 4. Linear Regression and Gradient Descent

The scratch model uses y_hat = Xw + b and minimizes mean squared error with batch gradient descent and tolerance-based early stopping.


## 5. Train and evaluate both targets


In [ ]:
def train_target(target_col):
    target_df = df.drop(columns=['Y2' if target_col == 'Y1' else 'Y1'])
    X_train, X_test, y_train, y_test, _, _ = preprocess_pipeline(
        target_df, target_col=target_col, test_size=0.2, random_seed=42
    )
    model = LinearRegressionScratch(learning_rate=0.01, n_iterations=10000, tolerance=1e-6)
    model.fit(X_train.values, y_train)
    prediction = model.predict(X_test.values)
    baseline = np.full(y_test.shape, np.mean(y_train))
    return {
        'model': model, 'y_test': y_test, 'prediction': prediction,
        'baseline': baseline, 'columns': X_train.columns,
    }

results = {target: train_target(target) for target in ['Y1', 'Y2']}
for target, result in results.items():
    print(target, 'MAE=', mae(result['y_test'], result['prediction']))
    print(target, 'RMSE=', rmse(result['y_test'], result['prediction']))
    print(target, 'R2=', r2_score(result['y_test'], result['prediction']))
    print(target, 'iterations=', result['model'].iterations_run)


## 6. Diagnostics and 5-fold cross-validation


In [ ]:
for target, result in results.items():
    residuals = result['y_test'] - result['prediction']
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].scatter(result['y_test'], result['prediction'], alpha=0.5)
    axes[0].set_title(f'{target}: actual vs predicted')
    axes[1].hist(residuals, bins=30, color='slateblue', edgecolor='white')
    axes[1].set_title(f'{target}: residual distribution')
    plt.tight_layout()
    plt.show()

for target in ['Y1', 'Y2']:
    target_df = df.drop(columns=['Y2' if target == 'Y1' else 'Y1'])
    folds = make_kfold_splits(target_df, target_col=target, n_splits=5, random_seed=42)
    fold_scores = []
    for train_df, test_df in folds:
        X_train, X_test, y_train, y_test, _, _ = preprocess_pipeline(
            target_df, target_col=target, train_df=train_df, test_df=test_df,
            test_size=0.2, random_seed=42
        )
        model = LinearRegressionScratch(learning_rate=0.01, n_iterations=10000, tolerance=1e-6)
        model.fit(X_train.values, y_train)
        fold_scores.append(r2_score(y_test, model.predict(X_test.values)))
    print(target, 'CV R2 mean/std=', np.mean(fold_scores), np.std(fold_scores))


## 7. Conclusion and limitations

The model is a strong interpretable baseline for both targets. It does not capture all nonlinear heat-transfer effects, so residual patterns and generalization should be considered before production use.
